In [ ]:
import os, torch, torch.nn as nn, torch.optim as optim
from torchvision import models
from data_preprocessing import get_data_loaders
from evaluation_metrics import evaluate_model_metrics, measure_inference_metrics,measure_model_size_and_flops

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
data_dir = "tiny-imagenet-200"
train_loader, val_loader = get_data_loaders(data_dir, batch_size=64, num_workers=4, image_size=224)
print("len(train_loader):", len(train_loader))
print("len(val_loader):", len(val_loader))

In [ ]:
def build_baseline_model(num_classes=200):
    model = models.mobilenet_v2(pretrained=True)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, num_classes)
    return model

model_baseline = build_baseline_model()
model_baseline = model_baseline.to(device)
print("Baseline MobileNetV2 model built.")

In [ ]:
def train_model(model, train_loader, num_epochs=10, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")
    return model

print("Starting training baseline model...")
model_baseline = train_model(model_baseline, train_loader, num_epochs=10, lr=0.001)
print("Training complete.")

In [ ]:
acc = evaluate_model_metrics(model_baseline, val_loader, device)
lat, thr, pwr, eng, edp = measure_inference_metrics(model_baseline, val_loader, device)
flops, params = measure_model_size_and_flops(model_baseline)

print(f"\nBaseline Accuracy:      {acc:.2f}%")
print(f"Latency:               {lat*1e3:.2f} ms/img")
print(f"Throughput:            {thr:.2f} imgs/s")
if pwr is not None:
    print(f"Avg GPU Power:          {pwr:.2f} W")
    print(f"Energy per img:         {eng:.4f} J")
    print(f"Energy‑Delay Product:   {edp:.6f} J·s")
print(f"Params:                {params/1e6:.2f} M")
if flops is not None:
    print(f"FLOPs:                 {flops:.2f} GFLOPs")

torch.save(baseline.state_dict(), "weights/mobilenetv2_fp32.pth")
print("✓ weights/mobilenetv2_fp32.pth saved")